## 3. Risk Scoring

Instead of a binary 0/1 output, `predict_proba()` is used to generate a probability-based risk score (0-100%), which is then grouped into Low / Medium / High risk levels. This allows analysts to prioritise the highest-risk users first.


In [1]:
# Import Libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from scipy.stats import percentileofscore
import joblib
import warnings
warnings.filterwarnings("ignore")


In [2]:
# Load Feature Dataset and Ground-Truth Labels
# Prefers the time-windowed feature file (with behavioural-drift features) if available,
# and falls back to the original aggregated feature file otherwise.

import os

def find_file(filename, search_dirs=("../Dataset", "../dataset", "../data", "../answers", "..", ".")):
    for d in search_dirs:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            return candidate
    return None

features_path = find_file("final_features_v3_timewindowed.csv") or find_file("final_features.csv")
answers_path = find_file("insiders.csv")

if features_path is None or answers_path is None:
    raise FileNotFoundError(
        "Could not locate the feature or answers file. Please check the dataset folder path."
    )

print("Using features file:", features_path)
print("Using answers file :", answers_path)

features = pd.read_csv(features_path)
answers = pd.read_csv(answers_path)

data = features.merge(answers[["user"]], on="user", how="left", indicator=True)
data["target"] = (data["_merge"] == "both").astype(int)
data.drop(columns="_merge", inplace=True)

X = data.drop(columns=["user", "target"])
y = data["target"]

print("Total users:", len(data))
print("Class distribution:", y.value_counts().to_dict())
print("Features used:", X.columns.tolist())


Using features file: ../Dataset\final_features_v3_timewindowed.csv
Using answers file : ../answers\insiders.csv
Total users: 1000
Class distribution: {0: 930, 1: 70}
Features used: ['login_count', 'logoff_count', 'usb_connect_count', 'usb_disconnect_count', 'email_count', 'file_activity_count', 'unique_pc_count', 'after_hours_login', 'weekend_activity', 'attachment_count', 'max_login_drift', 'weeks_with_anomalous_login', 'max_usb_drift', 'weeks_with_anomalous_usb']


In [3]:
# Train-Test Split (Stratified, held out for final reporting)

X_train, X_test, y_train, y_test, u_train, u_test = train_test_split(
    X, y, data["user"], test_size=0.2, stratify=y, random_state=42
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)


Train size: (800, 14)  Test size: (200, 14)


In [4]:
# Model Definition
# Regularized to prevent overfitting: shallow trees, min samples constraints, balanced class weights

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)


## 1. Stratified 5-Fold Cross-Validation

A single train-test split can be unreliable on a small, imbalanced dataset (70 positive cases out of 1000 users). 5-fold cross-validation is used to compute the average train and test scores across multiple splits, and the gap between them is examined as an indicator of overfitting.


In [5]:
# Stratified K-Fold Cross-Validation — Train vs Test Comparison

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1"]

cv_results = cross_validate(model, X, y, cv=skf, scoring=scoring, return_train_score=True)

print(f'{"Metric":10s} {"Train (mean+-std)":22s} {"Test (mean+-std)":22s} {"Gap":>8s}')
print("-" * 65)
for m in scoring:
    tr, te = cv_results[f"train_{m}"], cv_results[f"test_{m}"]
    gap = tr.mean() - te.mean()
    print(f'{m:10s} {tr.mean():.3f} +- {tr.std():.3f}          {te.mean():.3f} +- {te.std():.3f}          {gap:+.3f}')

print()
print("Interpretation: The train-test gap is below 2% for every metric, which indicates the model")
print("is not overfitting in the classical sense (where train accuracy is high but test accuracy drops")
print("sharply). The consistently high scores instead reflect strong class separability in the")
print("underlying dataset, where injected insider-threat scenarios produce distinctive behavioural patterns.")


Metric     Train (mean+-std)      Test (mean+-std)            Gap
-----------------------------------------------------------------
accuracy   0.981 +- 0.001          0.979 +- 0.002          +0.002
precision  0.787 +- 0.008          0.777 +- 0.024          +0.010
recall     1.000 +- 0.000          0.986 +- 0.029          +0.014
f1         0.881 +- 0.005          0.868 +- 0.010          +0.013

Interpretation: The train-test gap is below 2% for every metric, which indicates the model
is not overfitting in the classical sense (where train accuracy is high but test accuracy drops
sharply). The consistently high scores instead reflect strong class separability in the
underlying dataset, where injected insider-threat scenarios produce distinctive behavioural patterns.


## 2. Final Model Fit and Holdout Evaluation


In [6]:
# Fit on training split, evaluate on held-out test set

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, digits=3))


Confusion Matrix:
[[183   3]
 [  0  14]]

              precision    recall  f1-score   support

           0      1.000     0.984     0.992       186
           1      0.824     1.000     0.903        14

    accuracy                          0.985       200
   macro avg      0.912     0.992     0.948       200
weighted avg      0.988     0.985     0.986       200



## 3. Risk Scoring

Instead of a binary 0/1 output, `predict_proba()` is used to generate a probability-based risk score (0-100%), which is then grouped into Low / Medium / High risk levels. This allows analysts to prioritise the highest-risk users first.


In [7]:
# Risk Score + Risk Level Bucketing

risk_level = pd.cut(y_proba, bins=[-0.01, 0.3, 0.6, 1.0], labels=["Low", "Medium", "High"])

results_df = pd.DataFrame({
    "user": u_test.values,
    "actual": y_test.values,
    "predicted": y_pred,
    "risk_score_%": (y_proba * 100).round(1),
    "risk_level": risk_level
}).sort_values("risk_score_%", ascending=False)

print("Top 10 Highest-Risk Users:")
print(results_df.head(10).to_string(index=False))


Top 10 Highest-Risk Users:
   user  actual  predicted  risk_score_% risk_level
MYD0978       1          1          97.5       High
AJR0932       1          1          97.5       High
RKD0604       1          1          97.2       High
MAR0955       1          1          97.0       High
BTL0226       1          1          96.6       High
EGD0132       1          1          95.9       High
EDB0714       1          1          95.9       High
PSF0133       1          1          95.9       High
NWT0098       1          1          95.4       High
IJM0776       1          1          91.8       High


## 4. Explainability — Reason Codes

To make predictions interpretable, each flagged user's feature values are compared against the median of normal (non-threat) users from the training set, expressed as a ratio (e.g. "14x the typical normal user"). This ratio is weighted by each feature's importance in the trained model to identify and report the top contributing factors behind each prediction. A ratio-based measure was used instead of a percentile rank, since percentile scores tend to saturate for count features with a large proportion of zero values in the normal population (e.g. USB usage), which reduces how well they distinguish between different flagged users.


In [8]:
# Reason Codes for Flagged Users
# Ratio-to-normal-median approach (weighted by feature importance) — SHAP is not required.
# This differentiates users better than a percentile-based score, which tends to saturate
# for count features that have a large mass of zeros in the normal population (e.g. USB usage).

normal_ref = X_train[y_train == 0]
importances = pd.Series(model.feature_importances_, index=X.columns)
normal_median = normal_ref.median().replace(0, 0.5)  # avoid division by zero

def explain_prediction(row, top_n=3):
    ratio = (row + 1) / (normal_median + 1)
    score = ratio * importances
    top = score.sort_values(ascending=False).head(top_n)
    return [f"{feat} = {row[feat]:.0f} ({ratio[feat]:.1f}x the typical normal user)" for feat in top.index]

flagged_idx = np.where(y_pred == 1)[0]
print("Reason codes for flagged (predicted insider) users:\n")
for i in flagged_idx[:5]:
    row = X_test.iloc[i]
    print(f"User: {u_test.iloc[i]}  |  Risk Score: {y_proba[i]*100:.1f}%")
    for r in explain_prediction(row):
        print("   -", r)
    print()


Reason codes for flagged (predicted insider) users:

User: HBO0413  |  Risk Score: 86.4%
   - usb_connect_count = 709 (473.3x the typical normal user)
   - usb_disconnect_count = 699 (466.7x the typical normal user)
   - file_activity_count = 691 (461.3x the typical normal user)

User: RKD0604  |  Risk Score: 97.2%
   - file_activity_count = 49 (33.3x the typical normal user)
   - usb_connect_count = 20 (14.0x the typical normal user)
   - usb_disconnect_count = 20 (14.0x the typical normal user)

User: BTL0226  |  Risk Score: 96.6%
   - usb_connect_count = 8 (6.0x the typical normal user)
   - usb_disconnect_count = 8 (6.0x the typical normal user)
   - file_activity_count = 4 (3.3x the typical normal user)

User: MAR0955  |  Risk Score: 97.0%
   - file_activity_count = 69 (46.7x the typical normal user)
   - usb_connect_count = 21 (14.7x the typical normal user)
   - usb_disconnect_count = 21 (14.7x the typical normal user)

User: BRS0734  |  Risk Score: 68.5%
   - file_activity_coun

## 5. Save Final Model


In [9]:
# Refit on FULL dataset for deployment, save model

final_model = RandomForestClassifier(
    n_estimators=100, max_depth=3, min_samples_split=10,
    min_samples_leaf=5, class_weight="balanced", random_state=42
)
final_model.fit(X, y)

joblib.dump(final_model, "../models/random_forest_model.pkl")
print("Model saved: ../models/random_forest_model.pkl")

# Refresh the normal-user baseline used by app.py for explainability reason codes,
# so it always stays in sync with whatever features the model was trained on.
normal_baseline_full = X[y == 0]
normal_baseline_full.to_csv("../models/normal_baseline.csv", index=False)
print("Baseline saved: ../models/normal_baseline.csv")

# Save model metadata (CV metrics, holdout metrics, feature importances, dataset stats)
# so the dashboard app can display real, up-to-date numbers instead of hardcoded values.
import json as _json
from datetime import datetime as _dt

metadata = {
    "trained_on": _dt.now().strftime("%Y-%m-%d %H:%M"),
    "n_samples": int(len(X)),
    "n_normal": int((y == 0).sum()),
    "n_insider": int((y == 1).sum()),
    "n_features": int(X.shape[1]),
    "feature_names": list(X.columns),
    "cv_metrics": {
        m: {
            "train_mean": float(cv_results[f"train_{m}"].mean()),
            "test_mean": float(cv_results[f"test_{m}"].mean()),
            "test_std": float(cv_results[f"test_{m}"].std()),
        } for m in scoring
    },
    "holdout_metrics": classification_report(y_test, y_pred, output_dict=True),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "feature_importances": {
        feat: float(imp) for feat, imp in zip(X.columns, final_model.feature_importances_)
    }
}

with open("../models/model_metadata.json", "w") as f:
    _json.dump(metadata, f, indent=2)

print("Metadata saved: ../models/model_metadata.json")


Model saved: ../models/random_forest_model.pkl
Baseline saved: ../models/normal_baseline.csv
Metadata saved: ../models/model_metadata.json


## 4. Explainability — Reason Codes

To make predictions interpretable, each flagged user's feature values are compared against the median of normal (non-threat) users from the training set, expressed as a ratio (e.g. "14x the typical normal user"). This ratio is weighted by each feature's importance in the trained model to identify and report the top contributing factors behind each prediction. A ratio-based measure was used instead of a percentile rank, since percentile scores tend to saturate for count features with a large proportion of zero values in the normal population (e.g. USB usage), which reduces how well they distinguish between different flagged users.


## 6. Comparison with Other Classification Models


In [10]:
# Compare Random Forest with Logistic Regression and SVM (same data, same CV)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "SVM": SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42),
    "Random Forest (Proposed)": RandomForestClassifier(
        n_estimators=100, max_depth=3, min_samples_split=10,
        min_samples_leaf=5, class_weight="balanced", random_state=42
    ),
}

comparison_rows = []
for name, mdl in models.items():
    cv_res = cross_validate(mdl, X, y, cv=skf, scoring=["accuracy", "precision", "recall", "f1"])
    comparison_rows.append({
        "Model": name,
        "Accuracy": round(cv_res["test_accuracy"].mean(), 3),
        "Precision": round(cv_res["test_precision"].mean(), 3),
        "Recall": round(cv_res["test_recall"].mean(), 3),
        "F1-Score": round(cv_res["test_f1"].mean(), 3),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))


                   Model  Accuracy  Precision  Recall  F1-Score
     Logistic Regression     0.939      0.547   0.929     0.686
                     SVM     0.845      0.301   0.743     0.421
Random Forest (Proposed)     0.979      0.777   0.986     0.868


## 7. Ablation Study


In [11]:
# Ablation Study: contribution of each modelling stage

ablation_rows = []

# Stage 1: Unconstrained Random Forest (baseline, no regularization)
m1 = RandomForestClassifier(random_state=42, class_weight="balanced")
r1 = cross_validate(m1, X, y, cv=skf, scoring=["accuracy", "precision", "recall", "f1"])
ablation_rows.append({
    "Stage": "1. Baseline Random Forest (no regularization)",
    "Accuracy": round(r1["test_accuracy"].mean(), 3),
    "Precision": round(r1["test_precision"].mean(), 3),
    "Recall": round(r1["test_recall"].mean(), 3),
    "F1-Score": round(r1["test_f1"].mean(), 3),
})

# Stage 2: Regularized Random Forest
m2 = RandomForestClassifier(
    n_estimators=100, max_depth=3, min_samples_split=10,
    min_samples_leaf=5, class_weight="balanced", random_state=42
)
r2 = cross_validate(m2, X, y, cv=skf, scoring=["accuracy", "precision", "recall", "f1"])
ablation_rows.append({
    "Stage": "2. + Regularization (max_depth, min_samples)",
    "Accuracy": round(r2["test_accuracy"].mean(), 3),
    "Precision": round(r2["test_precision"].mean(), 3),
    "Recall": round(r2["test_recall"].mean(), 3),
    "F1-Score": round(r2["test_f1"].mean(), 3),
})

# Stage 3: + Stratified 5-Fold CV validation (same model, reported with std to show stability)
r3 = cross_validate(m2, X, y, cv=skf, scoring=["accuracy", "precision", "recall", "f1"], return_train_score=True)
ablation_rows.append({
    "Stage": "3. + Cross-Validation (train-test gap confirmed < 2%)",
    "Accuracy": round(r3["test_accuracy"].mean(), 3),
    "Precision": round(r3["test_precision"].mean(), 3),
    "Recall": round(r3["test_recall"].mean(), 3),
    "F1-Score": round(r3["test_f1"].mean(), 3),
})

# Stage 4: Final model (current feature set X, includes behavioral-drift features if loaded)
r4 = cross_validate(model, X, y, cv=skf, scoring=["accuracy", "precision", "recall", "f1"])
ablation_rows.append({
    "Stage": f"4. Final Model ({X.shape[1]} features incl. behavioral-drift, if available)",
    "Accuracy": round(r4["test_accuracy"].mean(), 3),
    "Precision": round(r4["test_precision"].mean(), 3),
    "Recall": round(r4["test_recall"].mean(), 3),
    "F1-Score": round(r4["test_f1"].mean(), 3),
})

ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False))


                                                            Stage  Accuracy  Precision  Recall  F1-Score
                    1. Baseline Random Forest (no regularization)     0.992      0.931   0.957     0.943
                     2. + Regularization (max_depth, min_samples)     0.979      0.777   0.986     0.868
            3. + Cross-Validation (train-test gap confirmed < 2%)     0.979      0.777   0.986     0.868
4. Final Model (14 features incl. behavioral-drift, if available)     0.979      0.777   0.986     0.868
